# Celda 1 - Checklist Conforme a los Archivos ```NPY```
Cargar ```.npy``` con Memmap

In [1]:
from pathlib import Path
import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

SEED = 123

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE = Path(r"C:\Users\leona\Documents\Thesis_Project_UACH\Temp\Dataset\features_mfcc_labeled")

X_train = np.load(BASE / "X_train.npy", mmap_mode="r")
y_train = np.load(BASE / "y_train.npy")
X_val   = np.load(BASE / "X_val.npy",   mmap_mode="r")
y_val   = np.load(BASE / "y_val.npy")
X_test  = np.load(BASE / "X_test.npy",  mmap_mode="r")
y_test  = np.load(BASE / "y_test.npy")

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)

# Chequeo de clases
u, c = np.unique(y_train, return_counts=True)
print("Distribución train:", dict(zip(u, c)))

X_train: (52551, 3, 32, 201) float16
y_train: (52551,) int64
X_val:   (11253, 3, 32, 201) float16
X_test:  (11308, 3, 32, 201) float16
Distribución train: {np.int64(0): np.int64(7368), np.int64(1): np.int64(6535), np.int64(2): np.int64(10668), np.int64(3): np.int64(27980)}


# Celda 2 - ```tf.data``` (con cast a float32 dentro del pipeline)

In [2]:
BATCH = 64

def make_ds(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(20000, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH)
    ds = ds.map(lambda a,b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_ds(X_train, y_train, training=True)
val_ds   = make_ds(X_val, y_val, training=False)
test_ds  = make_ds(X_test, y_test, training=False)

# Celda 3 - Class weights (para el desbalance)

Esto ayuda mucho para que no “adivine todo clase 3” (clase predominante).

In [3]:
num_classes = 4
counts = np.bincount(y_train, minlength=num_classes)
total = counts.sum()

# Peso inverso a frecuencia (simple y efectivo)
class_weight = {i: float(total / (num_classes * counts[i])) for i in range(num_classes) if counts[i] > 0}
print("counts:", counts)
print("class_weight:", class_weight)

counts: [ 7368  6535 10668 27980]
class_weight: {0: 1.7830822475570032, 1: 2.0103672532517214, 2: 1.2315101237345332, 3: 0.4695407433881344}


# Celda 4 - Modelo CNN baseline (channels_first)

In [ ]:
num_classes = 4
input_shape = (3, 32, 201)
act_fn = "sigmoid"

model = models.Sequential([
    layers.Input(shape=input_shape),
    # Keras por default usa channels_last; en este caso se forza channels_first
    layers.Conv2D(16, (3,3), padding="same", activation=act_fn, data_format="channels_first"),
    layers.BatchNormalization(axis=1),
    layers.MaxPool2D((2,2), data_format="channels_first"),

    layers.Conv2D(32, (3,3), padding="same", activation=act_fn, data_format="channels_first"),
    layers.BatchNormalization(axis=1),
    layers.MaxPool2D((2,2), data_format="channels_first"),

    layers.Conv2D(64, (3,3), padding="same", activation=act_fn, data_format="channels_first"),
    layers.BatchNormalization(axis=1),
    layers.MaxPool2D((2,2), data_format="channels_first"),

    layers.Conv2D(128, (3,3), padding="same", activation=act_fn, data_format="channels_first"),
    layers.BatchNormalization(axis=1),
    layers.GlobalAveragePooling2D(data_format="channels_first"),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_18 (Conv2D)              │ (None, 16, 32, 201)    │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 16, 32, 201)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 100)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 32, 16, 100)    │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 32, 16, 100)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 32, 8, 50)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 64, 8, 50)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 64, 8, 50)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 64, 4, 25)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 128, 4, 25)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 128, 4, 25)     │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_8      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,916 (417.64 KB)

 Trainable params: 106,436 (415.77 KB)

 Non-trainable params: 480 (1.88 KB)

# Celda 5 - Entrenar (con callbacks)

In [9]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max",
        patience=10, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", mode="max",
        patience=3, factor=0.5
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BASE / "cnn_mfcc_queenRec1.h5"),
        monitor="val_accuracy", mode="max",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weight,
    callbacks=callbacks
)

Epoch 1/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.7306 - loss: 0.6968

822/822 ━━━━━━━━━━━━━━━━━━━━ 272s 318ms/step - accuracy: 0.7884 - loss: 0.5678 - val_accuracy: 0.6309 - val_loss: 1.0741 - learning_rate: 0.0010
Epoch 2/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.8775 - loss: 0.3222

822/822 ━━━━━━━━━━━━━━━━━━━━ 278s 338ms/step - accuracy: 0.8809 - loss: 0.3237 - val_accuracy: 0.8019 - val_loss: 0.6817 - learning_rate: 0.0010
Epoch 3/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step - accuracy: 0.8889 - loss: 0.2832

822/822 ━━━━━━━━━━━━━━━━━━━━ 261s 318ms/step - accuracy: 0.9002 - loss: 0.2646 - val_accuracy: 0.8169 - val_loss: 0.6754 - learning_rate: 0.0010
Epoch 4/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 255s 311ms/step - accuracy: 0.9202 - loss: 0.2121 - val_accuracy: 0.7725 - val_loss: 1.0205 - learning_rate: 0.0010
Epoch 5/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step - accuracy: 0.9315 - loss: 0.1830

822/822 ━━━━━━━━━━━━━━━━━━━━ 264s 321ms/step - accuracy: 0.9315 - loss: 0.1832 - val_accuracy: 0.8261 - val_loss: 0.5850 - learning_rate: 0.0010
Epoch 6/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 259s 315ms/step - accuracy: 0.9383 - loss: 0.1594 - val_accuracy: 0.7714 - val_loss: 0.7687 - learning_rate: 0.0010
Epoch 7/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 231s 280ms/step - accuracy: 0.9401 - loss: 0.1538 - val_accuracy: 0.7438 - val_loss: 1.0130 - learning_rate: 0.0010
Epoch 8/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 229s 278ms/step - accuracy: 0.9478 - loss: 0.1327 - val_accuracy: 0.7900 - val_loss: 0.7941 - learning_rate: 0.0010
Epoch 9/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step - accuracy: 0.9638 - loss: 0.0888

...

822/822 ━━━━━━━━━━━━━━━━━━━━ 229s 278ms/step - accuracy: 0.9811 - loss: 0.0446 - val_accuracy: 0.8701 - val_loss: 0.6118 - learning_rate: 2.5000e-04
Epoch 15/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step - accuracy: 0.9806 - loss: 0.0432

822/822 ━━━━━━━━━━━━━━━━━━━━ 228s 278ms/step - accuracy: 0.9813 - loss: 0.0429 - val_accuracy: 0.8738 - val_loss: 0.5774 - learning_rate: 2.5000e-04
Epoch 16/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 228s 277ms/step - accuracy: 0.9816 - loss: 0.0418 - val_accuracy: 0.8435 - val_loss: 0.6623 - learning_rate: 2.5000e-04
Epoch 17/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 227s 276ms/step - accuracy: 0.9835 - loss: 0.0384 - val_accuracy: 0.8592 - val_loss: 0.6206 - learning_rate: 2.5000e-04
Epoch 18/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 228s 277ms/step - accuracy: 0.9851 - loss: 0.0345 - val_accuracy: 0.8647 - val_loss: 0.7093 - learning_rate: 2.5000e-04
Epoch 19/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - accuracy: 0.9869 - loss: 0.0289

822/822 ━━━━━━━━━━━━━━━━━━━━ 227s 276ms/step - accuracy: 0.9876 - loss: 0.0278 - val_accuracy: 0.8740 - val_loss: 0.6355 - learning_rate: 1.2500e-04
Epoch 20/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 226s 275ms/step - accuracy: 0.9899 - loss: 0.0234 - val_accuracy: 0.8725 - val_loss: 0.6506 - learning_rate: 1.2500e-04
Epoch 21/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 227s 276ms/step - accuracy: 0.9888 - loss: 0.0245 - val_accuracy: 0.8712 - val_loss: 0.6466 - learning_rate: 1.2500e-04
Epoch 22/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 229s 279ms/step - accuracy: 0.9899 - loss: 0.0230 - val_accuracy: 0.8661 - val_loss: 0.7406 - learning_rate: 1.2500e-04
Epoch 23/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step - accuracy: 0.9908 - loss: 0.0199

822/822 ━━━━━━━━━━━━━━━━━━━━ 228s 277ms/step - accuracy: 0.9912 - loss: 0.0200 - val_accuracy: 0.8764 - val_loss: 0.6454 - learning_rate: 6.2500e-05
Epoch 24/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 227s 276ms/step - accuracy: 0.9921 - loss: 0.0184 - val_accuracy: 0.8755 - val_loss: 0.6738 - learning_rate: 6.2500e-05
Epoch 25/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 237s 288ms/step - accuracy: 0.9915 - loss: 0.0188 - val_accuracy: 0.8719 - val_loss: 0.6634 - learning_rate: 6.2500e-05
Epoch 26/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 227s 276ms/step - accuracy: 0.9921 - loss: 0.0181 - val_accuracy: 0.8763 - val_loss: 0.6603 - learning_rate: 6.2500e-05
Epoch 27/40
821/822 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.9905 - loss: 0.0191

822/822 ━━━━━━━━━━━━━━━━━━━━ 232s 282ms/step - accuracy: 0.9921 - loss: 0.0176 - val_accuracy: 0.8771 - val_loss: 0.6462 - learning_rate: 3.1250e-05
Epoch 28/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 257s 276ms/step - accuracy: 0.9931 - loss: 0.0160 - val_accuracy: 0.8760 - val_loss: 0.6594 - learning_rate: 3.1250e-05
Epoch 29/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 227s 276ms/step - accuracy: 0.9930 - loss: 0.0161 - val_accuracy: 0.8743 - val_loss: 0.6652 - learning_rate: 3.1250e-05
Epoch 30/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 226s 275ms/step - accuracy: 0.9928 - loss: 0.0165 - val_accuracy: 0.8722 - val_loss: 0.7045 - learning_rate: 3.1250e-05
Epoch 31/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 226s 275ms/step - accuracy: 0.9932 - loss: 0.0157 - val_accuracy: 0.8754 - val_loss: 0.6740 - learning_rate: 1.5625e-05
Epoch 32/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 226s 275ms/step - accuracy: 0.9932 - loss: 0.0156 - val_accuracy: 0.8752 - val_loss: 0.6850 - learning_rate: 1.5625e-05
Epoch 33/40
822/822 ━━━━━━━━━━━━━━━━━━━━ 231s 

# Celda 6 - Evaluación en Test (matriz + macro-F1)

In [10]:
probs = model.predict(test_ds)
y_pred = np.argmax(probs, axis=1)

print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

177/177 ━━━━━━━━━━━━━━━━━━━━ 14s 74ms/step
Confusion matrix:
 [[1235   95   88  173]
 [  42 1131  147   93]
 [  50   87 1912  250]
 [  19  143  330 5513]]

Classification report:
               precision    recall  f1-score   support

           0     0.9175    0.7762    0.8410      1591
           1     0.7768    0.8004    0.7884      1413
           2     0.7719    0.8317    0.8007      2299
           3     0.9144    0.9181    0.9162      6005

    accuracy                         0.8658     11308
   macro avg     0.8452    0.8316    0.8366     11308
weighted avg     0.8687    0.8658    0.8662     11308

